# 🧪 Lab 05: Read the Generated Java

Welcome to the generated-code autopsy bay. The physical plan told us that Spark created a codegen stage; now we inspect the Java source produced for that stage.

**Mission Objective:** map a tiny `Filter → Project` pipeline to `GeneratedIteratorForCodegenStage...`, `processNext()`, the filter condition, and the projected arithmetic in Spark's generated Java.

**Deterministic Guardrail:** this lab does not benchmark or change codegen settings. It maps the same query from physical operators to the generated Java that executes them.


### Step 1: Define the diagnostic session
Adaptive execution is disabled so the stage and generated-code output remain easy to associate with the physical plan.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .master("local[2]")
    .appName("lab-05-read-the-generated-java")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:23:03 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:23:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/24 06:23:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen enabled: true


### Step 2: Build the small confession
The query contains exactly the two operations we want to find: a native filter `id % 7 = 0` and a native projection `id * 3 + 1`. The action makes the result real before we inspect the generated source.


In [2]:
query = (spark.range(0, 100_000)
    .where((F.col("id") % 7) == 0)
    .select((F.col("id") * 3 + 1).alias("x")))

print("Result preview:")
query.show(5, truncate=False)


Result preview:


+---+
|x  |
+---+
|1  |
|22 |
|43 |
|64 |
|85 |
+---+
only showing top 5 rows


### Step 3: Establish the physical-plan landmarks
Before reading Java, record the physical operators and their codegen marker. These are the names we will map into the generated program.


In [3]:
print("=== Physical plan ===")
query.explain("formatted")


=== Physical plan ===
== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 100000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((id#0L % 7) = 0)

(3) Project [codegen id : 1]
Output [1]: [((id#0L * 3) + 1) AS x#4L]
Input [1]: [id#0L]




### Step 4: Print the generated Java
`explain("codegen")` exposes the generated iterator and Java source for the eligible stage. Do not read it from top to bottom. Start with the class name, enter `processNext()`, and search for the filter and projection expressions.


In [4]:
print("=== Generated Java ===")
query.explain("codegen")


=== Generated Java ===
Found 1 WholeStageCodegen subtrees.
== Subtree 1 / 1 (maxMethodCodeSize:368; maxConstantPoolSize:211(0.32% used); numInnerClasses:0) ==
*(1) Project [((id#0L * 3) + 1) AS x#4L]
+- *(1) Filter ((id#0L % 7) = 0)
   +- *(1) Range (0, 100000, step=1, splits=2)

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Iterator[] inputs;
/* 009 */   private boolean range_initRange_0;
/* 010 */   private long range_nextIndex_0;
/* 011 */   private TaskContext range_taskContext_0;
/* 012 */   private InputMetrics range_inputMetrics_0;
/* 013 */   private long range_batchEnd_0;
/* 014 */   private long range_numElementsTodo_0;
/* 015 */   private

### Step 5: Read the confession as a translation
Use the generated output above as evidence and locate these landmarks:

- `GeneratedIteratorForCodegenStage...` — the generated iterator class.
- `processNext()` — the generated hot-path method.
- `id % 7 == 0` — the filter logic, represented with generated value/null variables.
- `id * 3 + 1` — the projection logic, which Spark 4.2 may emit through arithmetic helper calls such as multiplyExact and addExact, followed by row-writing code.
- `range_value`, `filter_value`, `project_value`, `isNull`, and `UnsafeRowWriter` — generated state that supports Spark's internal row and SQL NULL semantics.


In [5]:
print("=== Evidence checklist ===")
print("Filter expression: id % 7 = 0")
print("Projection expression: id * 3 + 1")
print("Generated-code view requested: explain('codegen')")
print("Inspect the generated output above for processNext() and the matching expression fragments.")


=== Evidence checklist ===
Filter expression: id % 7 = 0
Projection expression: id * 3 + 1
Generated-code view requested: explain('codegen')
Inspect the generated output above for processNext() and the matching expression fragments.


# 📊 Post-Lab Analysis: The Confession Is in `processNext()`

This lab moved the same query through the final translation step. The physical plan named `Filter` and `Project`; `explain("codegen")` exposed the generated iterator and the `processNext()` method where those operations become one executable Java path.

### 1. The Operators Disappear into the Loop

The generated source does not preserve the physical plan as two polite, separately labeled blocks. The filter condition, projection arithmetic, null checks, and row-writing logic are woven into the generated loop. The operators remain useful as conceptual landmarks, but their generic boundaries are no longer the shape of the hot path.

### 2. Generated Java Is Detailed Because the API Is Not

Names such as `range_value`, `filter_value`, `project_value`, `isNull`, and `UnsafeRowWriter` exist because Spark is also generating internal-row reads, SQL NULL handling, type-specific operations, and output writes. The DataFrame expression looked small; the execution contract is not.

### 3. Read It as Evidence

The useful skill is not memorizing generated variable names. It is moving between the physical operator, its Catalyst expression, and the generated Java fragment. For this case: `Filter → id % 7 = 0` and `Project → id * 3 + 1`.

The plan gave us the names. `processNext()` is where Spark wrote the program beside them.
